# Two-lane literature-retrieval pipeline (OpenAlex + Semantic Scholar)

This notebook implements **Phase A** and **Phase B** from `TWO_LANE_PIPELINE_IMPLEMENTATION_PLAN_FROM_REPORT.md`.

- **Phase A**: config surface, run artifacts under `runs/{run_id}/`, structured logging, and per-stage metrics.
- **Phase B**: LLM query planner that extracts **8–20 atomic bilingual facets** (EN+DE) with **strict JSON schema** output + caching.

**Ground rules for this test version**

- Prefer loud, detailed errors (so we can fix fast) over silent failure.
- Each code cell ends with a compact, easy-to-verify summary.
- When an OpenAI call happens, the cell prints token usage + estimated cost.


## Step 0 — Edit your chapter inputs

Edit the next cell, then run the notebook top-to-bottom.

What to look for (high level):

- **Good**: `runs/<run_id>/` created and contains `query_plan.json`, `logs.jsonl`, `metrics.json`.
- **Bad**: missing env vars (e.g. `OPENAI_API_KEY`) or schema errors from the query planner.


In [ ]:
# -----------------------------
# USER INPUTS (edit this cell)
# -----------------------------

chapter_title = "Analyse: Ökonomische Befunde im (west-)römischen Reich und ihre Beziehungen zu Politik, Gesellschaft und Militär"

chapter_spec_text = """
Dieses Kapitel untersucht wirtschaftliche Faktoren im (west-)römischen Reich der Spätantike und stellt die Befunde so dar, 
 dass sie für die Erklärung des Zerfalls- bzw. Transformationsprozesses nutzbar sind. Es arbeitet wirtschaftliche Mechanismen, 
 Strukturveränderungen und Rahmenbedingungen heraus und ordnet sie als Befundbestand, wobei Befund, 
 Deutung und Reichweite der Schlussfolgerungen getrennt ausgewiesen werden. Anschließend wird geprüft, 
 in welcher Weise die ökonomischen Befunde mit politischen, 
 sozialen und militärischen Entwicklungen im (west-)römischen Reich verknüpft werden können: welche Beziehungen das Material stützt,
  wo Wechselwirkungen plausibel sind und wo Verbindungen unsicher bleiben. Das Kapitel endet mit einer zusammenfassenden Einordnung, 
  auf welcher Ebene wirtschaftliche Faktoren im Gesamtprozess erklärungsrelevant erscheinen.
""".strip()

# Keep this stable unless you intentionally want to invalidate caches.
pipeline_version = "two_lane_v1"

# Set True to ignore cached query_plan.json and re-call the LLM.
FORCE_REBUILD_QUERY_PLAN = False


def _fmt_int(x) -> str:
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def _truncate(text: str, max_len: int = 180) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "…")


def print_section(title: str, width: int = 80, char: str = "=") -> None:
    line = char * width
    print(line)
    print(title)
    print(line)


def print_kv(d: dict, key_width: int = 26) -> None:
    for k, v in d.items():
        print(f"{str(k):<{key_width}} {v}")


print_section("User Inputs")
print_kv(
    {
        "chapter_title": _truncate(chapter_title),
        "chapter_spec_chars": _fmt_int(len(chapter_spec_text)),
        "pipeline_version": pipeline_version,
        "FORCE_REBUILD_QUERY_PLAN": FORCE_REBUILD_QUERY_PLAN,
    }
)


In [ ]:
# Phase A.0 — Imports + env loading

import os
import sys
import json
import time
import hashlib
import logging
import re
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

import importlib.metadata as importlib_metadata


def _fmt_int(x) -> str:
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def _truncate(text: str, max_len: int = 120) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "…")


def print_section(title: str, width: int = 80, char: str = "=") -> None:
    line = char * width
    print(line)
    print(title)
    print(line)


def print_kv(d: dict, key_width: int = 26) -> None:
    for k, v in d.items():
        print(f"{str(k):<{key_width}} {v}")


def print_table(rows, *, columns, max_rows: int = 200, max_col_width: int = 60) -> None:
    rows = list(rows or [])
    if not rows:
        print("<empty>")
        return

    show = rows[:max_rows]
    cols = list(columns)

    def cell(row, col):
        v = row.get(col, "")
        if v is None:
            v = ""
        s = str(v)
        if len(s) > max_col_width:
            s = s[: max_col_width - 1] + "…"
        return s

    widths = {c: max(len(str(c)), max(len(cell(r, c)) for r in show)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in show:
        print(" | ".join(f"{cell(r, c):<{widths[c]}}" for c in cols))

    if len(rows) > max_rows:
        print(f"... ({_fmt_int(len(rows) - max_rows)} more rows)")


def _pkg_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except Exception:
        return "unknown"


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    markers = ["package.json", "pyproject.toml", ".git", "README.md"]
    for d in [start, *start.parents]:
        if any((d / m).exists() for m in markers):
            return d
    return start


def find_notebook_path(repo_root: Path, notebook_filename: str) -> Optional[Path]:
    matches = sorted(repo_root.rglob(notebook_filename))
    if not matches:
        return None
    if len(matches) == 1:
        return matches[0]

    for m in matches:
        if m.parent.name.lower() == "sources-v2":
            return m
    return matches[0]


REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_FILENAME = "sources_two_lane.ipynb"

NOTEBOOK_PATH = find_notebook_path(REPO_ROOT, NOTEBOOK_FILENAME)
if NOTEBOOK_PATH is not None:
    NOTEBOOK_DIR = NOTEBOOK_PATH.parent.resolve()
else:
    fallback = REPO_ROOT / "sources-v2"
    NOTEBOOK_DIR = (fallback if fallback.exists() else Path.cwd()).resolve()

try:
    from dotenv import load_dotenv
except Exception as e:
    raise ImportError(
        "Missing dependency: python-dotenv. Install with: pip install python-dotenv"
    ) from e

# Load repo-root .env first (override=True so local project config wins),
# then load fastapi/.env as a fallback (override=False).
loaded_env_files = []
root_env = REPO_ROOT / ".env"
if root_env.exists():
    load_dotenv(dotenv_path=root_env, override=True)
    loaded_env_files.append(str(root_env))

fastapi_env = REPO_ROOT / "fastapi" / ".env"
if fastapi_env.exists():
    load_dotenv(dotenv_path=fastapi_env, override=False)
    loaded_env_files.append(str(fastapi_env))

ENV_VARS = [
    "OPENAI_API_KEY",
    "OPENALEX_API_KEY",
    "OPENALEX_EMAIL",
    "OPENALEX_MAILTO",
    "SEMANTICSCHOLAR_API_KEY",
]

env_status = {k: ("<set>" if (os.getenv(k) or "").strip() else "<missing>") for k in ENV_VARS}
missing = [k for k, v in env_status.items() if v == "<missing>"]

print_section("Phase A.0 — Environment")
print_kv(
    {
        "cwd": Path.cwd(),
        "repo_root": REPO_ROOT,
        "notebook_dir": NOTEBOOK_DIR,
        "dotenv_loaded": loaded_env_files if loaded_env_files else "<none found>",
    },
    key_width=14,
)

print("\nVersions:")
print_kv(
    {
        "python": sys.version.split()[0],
        "openai": _pkg_version("openai"),
        "pydantic": _pkg_version("pydantic"),
        "python-dotenv": _pkg_version("python-dotenv"),
    },
    key_width=14,
)

print("\nEnv vars (presence only):")
print_table(
    [{"env_var": k, "status": env_status[k]} for k in ENV_VARS],
    columns=["env_var", "status"],
    max_rows=50,
)

if missing:
    print("\nMissing:")
    for k in missing:
        print(f"- {k}")


In [ ]:
# Phase A.1 — Config + run artifacts + structured logging helpers

from typing import Any, Dict, Optional
from contextlib import contextmanager

from pydantic import BaseModel, Field
from pydantic.config import ConfigDict


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def stable_hash(*parts: str, length: int = 24) -> str:
    payload = "\n".join([(p or "").strip().replace("\r\n", "\n") for p in parts])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:length]


def compute_run_id(chapter_title: str, chapter_spec_text: str, pipeline_version: str) -> str:
    return stable_hash(pipeline_version, chapter_title, chapter_spec_text, length=24)


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def _json_default(o: Any):
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")


def write_json(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default) + "\n",
        encoding="utf-8",
    )
    tmp.replace(path)


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def append_jsonl(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=_json_default) + "\n")


class PipelineConfig(BaseModel):
    """Single config surface for the two-lane pipeline (test version)."""

    model_config = ConfigDict(extra="forbid")

    # Identity
    pipeline_version: str = "two_lane_v1"
    runs_root: Path

    # OpenAI (Phase B)
    openai_api_key: Optional[str] = Field(default=None, repr=False)
    openai_model_planner: str = "gpt-5-mini"
    openai_reasoning_effort: str = "high"
    openai_timeout_s: float = 120.0

    # Providers (later phases)
    openalex_base_url: str = "https://api.openalex.org"
    openalex_api_key: Optional[str] = Field(default=None, repr=False)
    openalex_email: Optional[str] = None
    openalex_timeout_s: float = 60.0
    openalex_rps: float = 10.0

    semanticscholar_base_url: str = "https://api.semanticscholar.org/graph/v1"
    semanticscholar_api_key: Optional[str] = Field(default=None, repr=False)
    semanticscholar_timeout_s: float = 60.0
    semanticscholar_rps: float = 1.0

    # Hard caps
    max_queries_per_provider: int = 50

    # Embeddings (later phases)
    embedding_model: str = "text-embedding-3-small"
    embedding_batch_size: int = 256

    # Pruning (later phases)
    prune_n1: int = 600

    # S2 neighbor booster (later phases)
    s2_neighbor_seed_count: int = 5
    s2_recs_limit_per_seed: int = 300

    # Rerank (later phases)
    rerank_top_k_pre: int = 40
    rerank_concurrency: int = 20

    # Match aggregation weights (later phases)
    match_weight_best: float = 0.55
    match_weight_top_m: float = 0.25
    match_weight_cov: float = 0.20
    match_m: int = 3

    # Scoring constants (later phases)
    scoring_t: float = 0.30
    scoring_t_noabs: float = 0.35

    # Authority time stratification (later phases)
    authority_classic_year_max: int = 2004
    authority_recent_year_window: int = 8
    authority_bucket_quotas: Dict[str, int] = Field(
        default_factory=lambda: {"classic": 8, "mid": 6, "recent": 6}
    )

    @classmethod
    def from_env(
        cls,
        *,
        repo_root: Path,
        notebook_dir: Optional[Path],
        pipeline_version: str,
    ) -> "PipelineConfig":
        base_dir = (notebook_dir or repo_root).resolve()
        return cls(
            pipeline_version=pipeline_version,
            runs_root=base_dir / "runs",
            openai_api_key=(os.getenv("OPENAI_API_KEY") or "").strip() or None,
            openalex_api_key=(os.getenv("OPENALEX_API_KEY") or "").strip() or None,
            openalex_email=(
                (os.getenv("OPENALEX_EMAIL") or "").strip()
                or (os.getenv("OPENALEX_MAILTO") or "").strip()
                or None
            ),
            semanticscholar_api_key=(os.getenv("SEMANTICSCHOLAR_API_KEY") or "").strip() or None,
        )

    def masked(self) -> Dict[str, Any]:
        d = self.model_dump(mode="python")
        for k in ["openai_api_key", "openalex_api_key", "semanticscholar_api_key"]:
            d[k] = "<set>" if d.get(k) else "<missing>"
        d["runs_root"] = str(d["runs_root"])
        return d


class RunArtifacts(BaseModel):
    model_config = ConfigDict(extra="forbid")

    query_plan_json: Path
    openalex_queries_json: Path
    semanticscholar_queries_json: Path

    openalex_raw_jsonl: Path
    semanticscholar_raw_jsonl: Path
    semanticscholar_recommendations_jsonl: Path

    candidates_normalized_jsonl: Path
    candidates_normalized_csv: Path

    embeddings_manifest_jsonl: Path
    embeddings_manifest_csv: Path
    embeddings_vectors_dir: Path

    rerank_results_jsonl: Path
    output_json: Path

    logs_jsonl: Path
    run_log: Path
    metrics_json: Path


class RunContext(BaseModel):
    model_config = ConfigDict(extra="forbid")

    repo_root: Path
    run_id: str
    run_dir: Path
    artifacts: RunArtifacts

    def create_artifact_skeleton(self, *, overwrite: bool = False) -> None:
        ensure_dir(self.run_dir)
        ensure_dir(self.artifacts.embeddings_vectors_dir)

        jsonl_files = [
            self.artifacts.openalex_raw_jsonl,
            self.artifacts.semanticscholar_raw_jsonl,
            self.artifacts.semanticscholar_recommendations_jsonl,
            self.artifacts.candidates_normalized_jsonl,
            self.artifacts.embeddings_manifest_jsonl,
            self.artifacts.rerank_results_jsonl,
            self.artifacts.logs_jsonl,
        ]
        for p in jsonl_files:
            ensure_dir(p.parent)
            p.touch(exist_ok=True)

        ensure_dir(self.artifacts.run_log.parent)
        self.artifacts.run_log.touch(exist_ok=True)

        csv_files = [
            self.artifacts.candidates_normalized_csv,
            self.artifacts.embeddings_manifest_csv,
        ]
        for p in csv_files:
            ensure_dir(p.parent)
            if overwrite or not p.exists():
                p.write_text("", encoding="utf-8")

        if overwrite or not self.artifacts.metrics_json.exists():
            write_json(
                self.artifacts.metrics_json,
                {
                    "run_id": self.run_id,
                    "created_at_utc": utc_now_iso(),
                    "stages": {},
                },
            )


def setup_run_logger(run_ctx: RunContext, *, level: int = logging.INFO) -> logging.Logger:
    ensure_dir(run_ctx.run_dir)

    logger = logging.getLogger("two_lane")
    logger.setLevel(level)
    logger.propagate = False

    # Idempotent re-runs in notebooks.
    for h in list(logger.handlers):
        logger.removeHandler(h)

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    sh = logging.StreamHandler(stream=sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(run_ctx.artifacts.run_log, encoding="utf-8")
    fh.setFormatter(fmt)

    logger.addHandler(sh)
    logger.addHandler(fh)
    return logger


def log_event(run_ctx: RunContext, *, stage: str, event: str, **fields: Any) -> None:
    rec = {"ts": utc_now_iso(), "stage": stage, "event": event, **fields}
    append_jsonl(run_ctx.artifacts.logs_jsonl, rec)

    lg = logging.getLogger("two_lane")
    level = logging.INFO
    if "error" in fields or event.endswith("_error") or event.endswith("_failed"):
        level = logging.ERROR

    lg.log(
        level,
        json.dumps(
            {k: v for k, v in rec.items() if k != "ts"},
            ensure_ascii=False,
            default=_json_default,
        ),
    )


def load_metrics(run_ctx: RunContext) -> Dict[str, Any]:
    try:
        return read_json(run_ctx.artifacts.metrics_json)
    except Exception:
        return {"run_id": run_ctx.run_id, "created_at_utc": utc_now_iso(), "stages": {}}


def save_metrics(run_ctx: RunContext, metrics: Dict[str, Any]) -> None:
    metrics = dict(metrics)
    metrics["updated_at_utc"] = utc_now_iso()
    write_json(run_ctx.artifacts.metrics_json, metrics)


@contextmanager
def stage_timer(run_ctx: RunContext, stage: str):
    t0 = time.time()
    yield
    dt = time.time() - t0
    metrics = load_metrics(run_ctx)
    metrics.setdefault("stages", {}).setdefault(stage, {})["last_duration_s"] = round(dt, 3)
    save_metrics(run_ctx, metrics)


In [ ]:
# Phase A.2 — Create run directory + artifact skeleton (no provider calls)

# Build config from env (runs are stored next to the notebook)
cfg = PipelineConfig.from_env(
    repo_root=REPO_ROOT,
    notebook_dir=NOTEBOOK_DIR,
    pipeline_version=pipeline_version,
)

# Compute run_id from chapter inputs
run_id = compute_run_id(chapter_title, chapter_spec_text, cfg.pipeline_version)
run_dir = cfg.runs_root / run_id

artifacts = RunArtifacts(
    query_plan_json=run_dir / "query_plan.json",
    openalex_queries_json=run_dir / "openalex_queries.json",
    semanticscholar_queries_json=run_dir / "semanticscholar_queries.json",
    openalex_raw_jsonl=run_dir / "openalex_raw.jsonl",
    semanticscholar_raw_jsonl=run_dir / "semanticscholar_raw.jsonl",
    semanticscholar_recommendations_jsonl=run_dir / "semanticscholar_recommendations.jsonl",
    candidates_normalized_jsonl=run_dir / "candidates_normalized.jsonl",
    candidates_normalized_csv=run_dir / "candidates_normalized.csv",
    embeddings_manifest_jsonl=run_dir / "embeddings_manifest.jsonl",
    embeddings_manifest_csv=run_dir / "embeddings_manifest.csv",
    embeddings_vectors_dir=run_dir / "embeddings_vectors",
    rerank_results_jsonl=run_dir / "rerank_results.jsonl",
    output_json=run_dir / "output.json",
    logs_jsonl=run_dir / "logs.jsonl",
    run_log=run_dir / "run.log",
    metrics_json=run_dir / "metrics.json",
)

run_ctx = RunContext(repo_root=REPO_ROOT, run_id=run_id, run_dir=run_dir, artifacts=artifacts)

with stage_timer(run_ctx, "phase_a"):
    run_ctx.create_artifact_skeleton(overwrite=False)
    logger = setup_run_logger(run_ctx)
    log_event(run_ctx, stage="phase_a", event="run_initialized", run_id=run_id, run_dir=str(run_dir))

    metrics = load_metrics(run_ctx)
    metrics.setdefault("stages", {}).setdefault("phase_a", {})["initialized_at_utc"] = utc_now_iso()
    save_metrics(run_ctx, metrics)

# Show current run folder contents
files = sorted([p.relative_to(run_dir) for p in run_dir.rglob("*") if p.is_file()])

print_section("Phase A.2 — Run Initialized")
print_kv({"run_id": run_id, "run_dir": run_dir, "runs_root": cfg.runs_root}, key_width=10)

print("\nConfig (masked):")
print_kv(cfg.masked(), key_width=28)

print("\nFiles currently in run_dir:")
print_table([{"file": str(f)} for f in files], columns=["file"], max_rows=200)


In [ ]:
# Phase B.1 — Data models (strict) for the Query Planner output

from typing import List


class BilingualTerms(BaseModel):
    model_config = ConfigDict(extra="forbid")

    en: List[str] = Field(default_factory=list)
    de: List[str] = Field(default_factory=list)


class Facet(BaseModel):
    model_config = ConfigDict(extra="forbid")

    facet_id: str
    facet_label_en: str
    facet_label_de: str
    facet_type: str
    importance_weight: int = Field(ge=1, le=5)
    text_en: str
    text_de: str
    canonical_terms: BilingualTerms
    neighbor_terms: BilingualTerms
    exclusion_terms: BilingualTerms


class QueryPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    topic_summary_en: str
    topic_summary_de: str
    facets: List[Facet]
    global_canonical_terms: BilingualTerms
    global_exclusions: BilingualTerms


class ChapterInput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    chapter_title: str
    chapter_spec_text: str
    pipeline_version: str

    def compute_run_id(self) -> str:
        return compute_run_id(self.chapter_title, self.chapter_spec_text, self.pipeline_version)


In [ ]:
# Phase B.2 — OpenAI helpers: strict JSON schema outputs + token/cost tracking

import json
import time
import re
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

from openai import OpenAI


# Model prices (USD per *1M tokens*) — verify periodically.
# NOTE: cached token pricing can differ; set the correct values for your account.
MODEL_PRICES_USD_PER_1M: Dict[str, Dict[str, float]] = {
    "gpt-5.2": {"input": 1.75, "cached": 1.75, "output": 14.00},
    "gpt-5-mini": {"input": 0.25, "cached": 0.25, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "cached": 0.05, "output": 0.40},
}


def resolve_pricing_key(model_name: str) -> Optional[str]:
    model_lower = (model_name or "").lower()
    normalized = {k.lower(): k for k in MODEL_PRICES_USD_PER_1M}

    if model_lower in normalized:
        return normalized[model_lower]

    # Strip trailing release suffix: -YYYY-MM-DD
    date_stripped = re.sub(r"-20\d{2}-\d{2}-\d{2}$", "", model_lower)
    if date_stripped in normalized:
        return normalized[date_stripped]

    # Prefix match: gpt-5.2-foo
    for k_lower, original in normalized.items():
        if model_lower.startswith(k_lower + "-"):
            return original

    return None


def extract_usage(response) -> Dict[str, int]:
    usage = getattr(response, "usage", None)
    if usage is None:
        return {
            "input_tokens": 0,
            "output_tokens": 0,
            "reasoning_tokens": 0,
            "cached_input_tokens": 0,
        }

    input_tokens = int(getattr(usage, "input_tokens", None) or getattr(usage, "prompt_tokens", None) or 0)
    output_tokens = int(
        getattr(usage, "output_tokens", None) or getattr(usage, "completion_tokens", None) or 0
    )

    cached_input_tokens = 0
    input_details = getattr(usage, "input_tokens_details", None) or getattr(usage, "prompt_tokens_details", None)
    if input_details is not None:
        cached_input_tokens = int(getattr(input_details, "cached_tokens", 0) or 0)

    reasoning_tokens = 0
    out_details = getattr(usage, "output_tokens_details", None) or getattr(usage, "completion_tokens_details", None)
    if out_details is not None:
        reasoning_tokens = int(getattr(out_details, "reasoning_tokens", 0) or 0)

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "cached_input_tokens": cached_input_tokens,
    }


def estimate_cost_usd(
    *,
    model_used: str,
    input_tokens: int,
    cached_input_tokens: int,
    output_tokens: int,
) -> Dict[str, Any]:
    key = resolve_pricing_key(model_used)
    if key is None:
        return {
            "pricing_key": None,
            "input_cost_usd": 0.0,
            "cached_input_cost_usd": 0.0,
            "output_cost_usd": 0.0,
            "total_cost_usd": 0.0,
            "note": f"No pricing found for model={model_used!r}; update MODEL_PRICES_USD_PER_1M.",
        }

    prices = MODEL_PRICES_USD_PER_1M[key]
    in_price = float(prices.get("input", 0.0) or 0.0)
    cached_price = float(prices.get("cached", in_price) or in_price)
    out_price = float(prices.get("output", 0.0) or 0.0)

    cached_input_tokens = max(0, min(int(cached_input_tokens or 0), int(input_tokens or 0)))
    billable_input_tokens = max(int(input_tokens or 0) - cached_input_tokens, 0)

    input_cost = (billable_input_tokens / 1_000_000) * in_price
    cached_cost = (cached_input_tokens / 1_000_000) * cached_price
    output_cost = (int(output_tokens or 0) / 1_000_000) * out_price

    return {
        "pricing_key": key,
        "price_per_million": {"input": in_price, "cached": cached_price, "output": out_price},
        "billable_input_tokens": billable_input_tokens,
        "input_cost_usd": float(input_cost),
        "cached_input_cost_usd": float(cached_cost),
        "output_cost_usd": float(output_cost),
        "total_cost_usd": float(input_cost + cached_cost + output_cost),
        "note": "output_tokens already includes reasoning_tokens; reasoning_tokens is shown for diagnostics only.",
    }


def _response_to_jsonable(response) -> Dict[str, Any]:
    if response is None:
        return {}

    if hasattr(response, "model_dump"):
        try:
            return response.model_dump(mode="json")
        except TypeError:
            return response.model_dump()
        except Exception:
            pass

    if hasattr(response, "to_dict"):
        try:
            return response.to_dict()
        except Exception:
            pass

    return {"repr": repr(response)}


def extract_output_text_or_refusal(response) -> Tuple[str, Optional[str]]:
    text = getattr(response, "output_text", None)
    if isinstance(text, str) and text.strip():
        return text, None

    parts = []
    refusal = None

    output_items = getattr(response, "output", None) or []
    for item in output_items:
        item_type = item.get("type") if isinstance(item, dict) else getattr(item, "type", None)
        if item_type != "message":
            continue

        content_items = item.get("content") if isinstance(item, dict) else getattr(item, "content", None)
        for c in content_items or []:
            c_type = c.get("type") if isinstance(c, dict) else getattr(c, "type", None)
            if c_type == "output_text":
                t = c.get("text") if isinstance(c, dict) else getattr(c, "text", None)
                if t:
                    parts.append(str(t))
            elif c_type == "refusal":
                r = c.get("refusal") if isinstance(c, dict) else getattr(c, "refusal", None)
                if r:
                    refusal = (str(refusal) + "\n" + str(r)) if refusal else str(r)

    joined = "\n".join([p for p in parts if str(p).strip()])
    return joined, refusal


def openai_json_schema_call(
    *,
    api_key: str,
    model: str,
    system_prompt: str,
    user_prompt: str,
    schema_name: str,
    schema: Dict[str, Any],
    reasoning_effort: str = "high",
    max_output_tokens: int = 2000,
    timeout_s: float = 120.0,
    debug_dir: Optional[Path] = None,
    debug_prefix: str = "openai",
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    if not api_key:
        raise ValueError("OPENAI_API_KEY is missing. Set it in your repo .env (or environment) before Phase B.")

    client = OpenAI(api_key=api_key)

    t0 = time.time()
    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "input_text", "text": user_prompt}]},
        ],
        reasoning={"effort": reasoning_effort},
        max_output_tokens=max_output_tokens,
        text={
            "format": {
                "type": "json_schema",
                "name": schema_name,
                "schema": schema,
                "strict": True,
            }
        },
        timeout=timeout_s,
    )
    dt = time.time() - t0

    model_used = getattr(response, "model", None) or model
    usage = extract_usage(response)
    cost = estimate_cost_usd(
        model_used=model_used,
        input_tokens=usage["input_tokens"],
        cached_input_tokens=usage["cached_input_tokens"],
        output_tokens=usage["output_tokens"],
    )

    response_id = getattr(response, "id", None)

    # Debug dumps (always when debug_dir is provided)
    if debug_dir is not None:
        try:
            ensure_dir(debug_dir)
            (debug_dir / f"{debug_prefix}.system_prompt.txt").write_text(system_prompt, encoding="utf-8")
            (debug_dir / f"{debug_prefix}.user_prompt.txt").write_text(user_prompt, encoding="utf-8")
            write_json(debug_dir / f"{debug_prefix}.response.json", _response_to_jsonable(response))
        except Exception:
            pass

    raw_text, refusal = extract_output_text_or_refusal(response)

    if debug_dir is not None:
        try:
            (debug_dir / f"{debug_prefix}.output_text.txt").write_text(raw_text or "", encoding="utf-8")
            if refusal:
                (debug_dir / f"{debug_prefix}.refusal.txt").write_text(refusal, encoding="utf-8")
        except Exception:
            pass

    if refusal and not (raw_text or "").strip():
        raise ValueError(f"OpenAI refused the request: {refusal}")

    if not (raw_text or "").strip():
        raise ValueError("OpenAI response had no output_text.")

    try:
        obj = json.loads(raw_text)
    except Exception as e:
        raise ValueError(
            "Failed to parse JSON from OpenAI output_text. Inspect the debug artifacts saved in the run folder."
        ) from e

    meta = {
        "model_requested": model,
        "model_used": model_used,
        "response_id": response_id,
        "latency_s": round(dt, 3),
        "usage": usage,
        "cost_estimate": cost,
    }
    return obj, meta


In [ ]:
# Phase B.3 — LLM Query Planner (facet extraction; atomic bilingual facets)

import traceback
from typing import Any, Dict, Tuple


QUERY_PLAN_JSON_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "topic_summary_en",
        "topic_summary_de",
        "facets",
        "global_canonical_terms",
        "global_exclusions",
    ],
    "properties": {
        "topic_summary_en": {"type": "string"},
        "topic_summary_de": {"type": "string"},
        "global_canonical_terms": {
            "type": "object",
            "additionalProperties": False,
            "required": ["en", "de"],
            "properties": {
                "en": {"type": "array", "items": {"type": "string"}},
                "de": {"type": "array", "items": {"type": "string"}},
            },
        },
        "global_exclusions": {
            "type": "object",
            "additionalProperties": False,
            "required": ["en", "de"],
            "properties": {
                "en": {"type": "array", "items": {"type": "string"}},
                "de": {"type": "array", "items": {"type": "string"}},
            },
        },
        "facets": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "facet_id",
                    "facet_label_en",
                    "facet_label_de",
                    "facet_type",
                    "importance_weight",
                    "text_en",
                    "text_de",
                    "canonical_terms",
                    "neighbor_terms",
                    "exclusion_terms",
                ],
                "properties": {
                    "facet_id": {"type": "string"},
                    "facet_label_en": {"type": "string"},
                    "facet_label_de": {"type": "string"},
                    "facet_type": {"type": "string"},
                    "importance_weight": {"type": "integer", "minimum": 1, "maximum": 5},
                    "text_en": {"type": "string"},
                    "text_de": {"type": "string"},
                    "canonical_terms": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["en", "de"],
                        "properties": {
                            "en": {"type": "array", "items": {"type": "string"}},
                            "de": {"type": "array", "items": {"type": "string"}},
                        },
                    },
                    "neighbor_terms": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["en", "de"],
                        "properties": {
                            "en": {"type": "array", "items": {"type": "string"}},
                            "de": {"type": "array", "items": {"type": "string"}},
                        },
                    },
                    "exclusion_terms": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["en", "de"],
                        "properties": {
                            "en": {"type": "array", "items": {"type": "string"}},
                            "de": {"type": "array", "items": {"type": "string"}},
                        },
                    },
                },
            },
        },
    },
}


PLANNER_SYSTEM_PROMPT = """You extract atomic, retrieval-oriented facets from a chapter specification.

Hard rules:
- Output MUST be valid JSON matching the provided schema.
- Do NOT name specific papers, authors, or venues.
- Produce 8–20 facets.
- Bilingual always: fill EN + DE fields.
- Atomic facet rule: each facet maps to exactly ONE requirement/aspect.
- Discourage overlap: facets should be as non-overlapping as possible.
- Add 2–4 neighbor facets that are not explicitly stated but commonly required.
- Prefer technical terms over vague words.
- If ambiguous terms exist, add exclusion_terms to disambiguate.
- Keep term lists focused (canonical_terms are the strongest; neighbor_terms are weaker).

Be deterministic"""


def planner_user_prompt(chapter_input: ChapterInput) -> str:
    return (
        "CHAPTER_TITLE:\n"
        + chapter_input.chapter_title.strip()
        + "\n\nCHAPTER_SPEC (raw):\n"
        + chapter_input.chapter_spec_text.strip()
        + "\n"
    )


def diagnose_query_plan(plan: QueryPlan) -> Dict[str, Any]:
    issues = []

    n_facets = len(plan.facets)
    if n_facets < 8 or n_facets > 20:
        issues.append(f"Facet count is {n_facets} (expected 8–20).")

    ids = [f.facet_id for f in plan.facets]
    dup_ids = sorted({x for x in ids if ids.count(x) > 1})
    if dup_ids:
        issues.append(f"Duplicate facet_id(s): {dup_ids}")

    bad_weights = [f.facet_id for f in plan.facets if not (1 <= f.importance_weight <= 5)]
    if bad_weights:
        issues.append(f"Facets with invalid importance_weight: {bad_weights}")

    # Very rough overlap heuristic: identical canonical term sets.
    canon_sets = {}
    overlaps = []
    for f in plan.facets:
        key = (
            tuple(sorted({t.strip().lower() for t in f.canonical_terms.en if t.strip()})),
            tuple(sorted({t.strip().lower() for t in f.canonical_terms.de if t.strip()})),
        )
        if key in canon_sets and key != ((), ()):
            overlaps.append((canon_sets[key], f.facet_id))
        else:
            canon_sets[key] = f.facet_id

    if overlaps:
        issues.append(f"Potential duplicate facets (identical canonical_terms): {overlaps[:5]}")

    return {"facet_count": n_facets, "issues": issues}


def _is_placeholder_cache(obj: Any) -> bool:
    if not isinstance(obj, dict):
        return False
    meta = obj.get("_meta")
    return isinstance(meta, dict) and meta.get("placeholder") is True


def plan_queries_llm(
    chapter_input: ChapterInput,
    *,
    config: PipelineConfig,
    run_ctx: RunContext,
    force_rebuild: bool = False,
) -> Tuple[QueryPlan, Dict[str, Any]]:
    stage = "phase_b_query_planner"
    cache_path = run_ctx.artifacts.query_plan_json

    if cache_path.exists() and not force_rebuild:
        try:
            cached_obj = read_json(cache_path)
            if _is_placeholder_cache(cached_obj):
                log_event(run_ctx, stage=stage, event="cache_placeholder_ignored", path=str(cache_path))
            else:
                plan = QueryPlan.model_validate(cached_obj)
                meta = {
                    "cache_hit": True,
                    "usage": {
                        "input_tokens": 0,
                        "cached_input_tokens": 0,
                        "output_tokens": 0,
                        "reasoning_tokens": 0,
                    },
                    "cost_estimate": {"total_cost_usd": 0.0},
                    "diagnostics": diagnose_query_plan(plan),
                }
                log_event(run_ctx, stage=stage, event="cache_hit", path=str(cache_path))
                return plan, meta
        except Exception as e:
            err = str(e)
            err_short = err if len(err) <= 800 else (err[:800] + "…")
            write_json(
                run_ctx.run_dir / "query_plan.cache_invalid.json",
                {"ts": utc_now_iso(), "path": str(cache_path), "error": err},
            )
            log_event(run_ctx, stage=stage, event="cache_invalid", path=str(cache_path), error=err_short)

    user_prompt = planner_user_prompt(chapter_input)

    with stage_timer(run_ctx, stage):
        try:
            obj, meta = openai_json_schema_call(
                api_key=config.openai_api_key or "",
                model=config.openai_model_planner,
                system_prompt=PLANNER_SYSTEM_PROMPT,
                user_prompt=user_prompt,
                schema_name="query_plan",
                schema=QUERY_PLAN_JSON_SCHEMA,
                reasoning_effort=config.openai_reasoning_effort,
                max_output_tokens=2000,
                timeout_s=config.openai_timeout_s,
                debug_dir=run_ctx.run_dir,
                debug_prefix="query_plan",
            )
        except Exception as e:
            dbg = {
                "ts": utc_now_iso(),
                "stage": stage,
                "error": str(e),
                "traceback": traceback.format_exc(),
                "chapter_title": chapter_input.chapter_title,
                "pipeline_version": chapter_input.pipeline_version,
            }
            write_json(run_ctx.run_dir / "query_plan.error.json", dbg)
            log_event(run_ctx, stage=stage, event="openai_error", error=str(e))
            raise

    # Persist raw object + OpenAI meta for debugging
    write_json(run_ctx.run_dir / "query_plan.raw_output.json", obj)
    write_json(run_ctx.run_dir / "query_plan.openai_meta.json", meta)

    try:
        plan = QueryPlan.model_validate(obj)
    except Exception as e:
        log_event(
            run_ctx,
            stage=stage,
            event="schema_validation_failed",
            error=str(e),
            raw_path=str(run_ctx.run_dir / "query_plan.raw_output.json"),
        )
        raise

    write_json(cache_path, plan.model_dump(mode="json"))
    log_event(
        run_ctx,
        stage=stage,
        event="cache_write",
        path=str(cache_path),
        model_used=meta.get("model_used"),
        usage=meta.get("usage"),
        cost=meta.get("cost_estimate"),
    )

    metrics = load_metrics(run_ctx)
    metrics.setdefault("stages", {}).setdefault(stage, {})["openai"] = meta
    save_metrics(run_ctx, metrics)

    meta = dict(meta)
    meta["cache_hit"] = False
    meta["diagnostics"] = diagnose_query_plan(plan)
    return plan, meta


In [ ]:
# Phase B.4 — Run the planner + inspect facets

try:
    import pandas as pd
except Exception:
    pd = None
    print("[WARN] pandas not available; facet table will be plain text. Install with: pip install pandas")

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None
    print("[WARN] matplotlib not available; facet plot will be skipped. Install with: pip install matplotlib")


chapter_input = ChapterInput(
    chapter_title=chapter_title,
    chapter_spec_text=chapter_spec_text,
    pipeline_version=pipeline_version,
)

expected_run_id = chapter_input.compute_run_id()
if expected_run_id != run_ctx.run_id:
    print_section("Phase B.4 — Warning")
    print_kv(
        {
            "msg": "run_id mismatch; rebuilding Phase A context with current inputs",
            "expected_run_id": expected_run_id,
            "current_run_id": run_ctx.run_id,
        },
        key_width=16,
    )

    run_id = expected_run_id
    run_dir = cfg.runs_root / run_id

    artifacts = artifacts.model_copy(
        update={
            "query_plan_json": run_dir / "query_plan.json",
            "openalex_queries_json": run_dir / "openalex_queries.json",
            "semanticscholar_queries_json": run_dir / "semanticscholar_queries.json",
            "openalex_raw_jsonl": run_dir / "openalex_raw.jsonl",
            "semanticscholar_raw_jsonl": run_dir / "semanticscholar_raw.jsonl",
            "semanticscholar_recommendations_jsonl": run_dir / "semanticscholar_recommendations.jsonl",
            "candidates_normalized_jsonl": run_dir / "candidates_normalized.jsonl",
            "candidates_normalized_csv": run_dir / "candidates_normalized.csv",
            "embeddings_manifest_jsonl": run_dir / "embeddings_manifest.jsonl",
            "embeddings_manifest_csv": run_dir / "embeddings_manifest.csv",
            "embeddings_vectors_dir": run_dir / "embeddings_vectors",
            "rerank_results_jsonl": run_dir / "rerank_results.jsonl",
            "output_json": run_dir / "output.json",
            "logs_jsonl": run_dir / "logs.jsonl",
            "run_log": run_dir / "run.log",
            "metrics_json": run_dir / "metrics.json",
        }
    )

    run_ctx = RunContext(repo_root=REPO_ROOT, run_id=run_id, run_dir=run_dir, artifacts=artifacts)
    run_ctx.create_artifact_skeleton(overwrite=False)
    logger = setup_run_logger(run_ctx)


plan, meta = plan_queries_llm(
    chapter_input,
    config=cfg,
    run_ctx=run_ctx,
    force_rebuild=FORCE_REBUILD_QUERY_PLAN,
)

diag = meta.get("diagnostics") or diagnose_query_plan(plan)

print_section("Phase B.4 — Query Plan")
print_kv(
    {
        "cache_hit": meta.get("cache_hit"),
        "run_id": run_ctx.run_id,
        "run_dir": run_ctx.run_dir,
        "query_plan.json": run_ctx.artifacts.query_plan_json,
    },
    key_width=14,
)

print_section("Topic Summary (EN)")
print(plan.topic_summary_en)

print_section("Topic Summary (DE)")
print(plan.topic_summary_de)

# OpenAI usage + cost
print_section("OpenAI Usage / Cost")
if meta.get("cache_hit"):
    print("cache hit — no new tokens billed")
else:
    usage = meta.get("usage") or {}
    cost = meta.get("cost_estimate") or {}

    input_tokens = int(usage.get("input_tokens") or 0)
    cached_input_tokens = int(usage.get("cached_input_tokens") or 0)
    billable_input_tokens = max(input_tokens - min(cached_input_tokens, input_tokens), 0)

    print_kv(
        {
            "model_used": meta.get("model_used"),
            "latency_s": meta.get("latency_s"),
            "input_tokens": _fmt_int(input_tokens),
            "cached_input_tokens": _fmt_int(cached_input_tokens),
            "billable_input_tokens": _fmt_int(billable_input_tokens),
            "output_tokens": _fmt_int(int(usage.get("output_tokens") or 0)),
            "reasoning_tokens": _fmt_int(int(usage.get("reasoning_tokens") or 0)),
            "est_total_cost_usd": f"{float(cost.get('total_cost_usd') or 0.0):.6f}",
        },
        key_width=20,
    )

    print("\nModel prices (USD per 1M tokens) — verify periodically:")
    for m, p in MODEL_PRICES_USD_PER_1M.items():
        print(f"- {m}: in={p.get('input')} cached={p.get('cached')} out={p.get('output')}")

    if cost.get("price_per_million"):
        print("\nPricing used (USD per 1M tokens):")
        p = cost["price_per_million"]
        print(f"- input={p.get('input')} cached={p.get('cached')} output={p.get('output')}")

    print("\nCost breakdown (USD):")
    print_kv(
        {
            "input_cost_usd": f"{float(cost.get('input_cost_usd') or 0.0):.6f}",
            "cached_input_cost_usd": f"{float(cost.get('cached_input_cost_usd') or 0.0):.6f}",
            "output_cost_usd": f"{float(cost.get('output_cost_usd') or 0.0):.6f}",
            "total_cost_usd": f"{float(cost.get('total_cost_usd') or 0.0):.6f}",
            "pricing_key": cost.get("pricing_key"),
        },
        key_width=24,
    )

print_section("Facet Diagnostics")
print_kv(
    {
        "facet_count": diag.get("facet_count"),
        "issues": len(diag.get("issues") or []),
    },
    key_width=12,
)
if diag.get("issues"):
    print("\nIssues:")
    for x in diag["issues"]:
        print(f"- {x}")

rows = []
for f in plan.facets:
    rows.append(
        {
            "facet_id": f.facet_id,
            "weight": f.importance_weight,
            "facet_type": f.facet_type,
            "label_en": _truncate(f.facet_label_en, 80),
            "canon_en": len(f.canonical_terms.en),
            "canon_de": len(f.canonical_terms.de),
            "excl_en": len(f.exclusion_terms.en),
            "excl_de": len(f.exclusion_terms.de),
        }
    )

print_section("Facets")
if pd is not None:
    df = pd.DataFrame(rows).sort_values(["weight", "facet_id"], ascending=[False, True])
    display(df)
else:
    print_table(
        sorted(rows, key=lambda x: (-x["weight"], x["facet_id"])),
        columns=["facet_id", "weight", "facet_type", "label_en", "canon_en", "canon_de", "excl_en", "excl_de"],
        max_rows=200,
    )

if plt is not None:
    plt.figure(figsize=(10, max(3, 0.35 * len(rows))))
    ordered = sorted(rows, key=lambda x: (-x["weight"], x["facet_id"]))
    ids = [r["facet_id"] for r in ordered]
    ws = [r["weight"] for r in ordered]
    plt.barh(ids, ws)
    plt.gca().invert_yaxis()
    plt.title("Facet importance weights")
    plt.xlabel("importance_weight (1–5)")
    plt.tight_layout()
    plt.show()
